# Module 8 - Lab 3: Build a SQL Database Agent using LangChain

**Module:** 8 - Advanced LLM Systems &nbsp;|&nbsp; **Scenario:** IT Operations Management &nbsp;|&nbsp; **Database:** Supabase (PostgreSQL)

In this lab you connect an **AI agent** to a **cloud PostgreSQL database**. You type requests in plain English, for example *"Show me all critical incidents"*. The agent chooses an authorized Python tool. The tool validates the input with **Pydantic** and runs a **parameterized SQL** statement against your database. The agent then explains the result in business language.

**Before you start**
- Run the cells **in order, from top to bottom**. Do not skip the setup cells.
- Your database password and OpenAI key belong **only** in your `.env` file. Never paste them into a cell, a chat or a shared file.
- Sections 1 to 7 need only the database. Sections 8 to 11 also need `OPENAI_API_KEY` in `.env`.
- Expected results are described in text. Your actual output comes from your own database.

## Section 1 - Introduction to SQL Database Agents

**What is a SQL database agent?** It is an AI agent that answers business questions by reading and changing a database through a fixed set of tools. The learner asks in plain English. The agent decides which tool to call, the tool runs an approved database operation, and the agent explains the result.

**Normal SQL application vs. AI database agent**

| | Normal SQL application | AI database agent |
|---|---|---|
| Who writes the query? | A developer writes fixed SQL in advance | The LLM picks a tool at runtime; the tool builds safe, parameterized SQL |
| How does a user ask? | Forms, buttons, fixed reports | Natural language |
| Who controls what runs? | The application code | The tools you define, plus database permissions |
| Main risk | Bugs in fixed queries | The LLM asking for something it should not do, so tools and validation must limit it |

**How LangChain connects an LLM to custom tools.** LangChain gives the LLM a list of tools, each with a name, a description and an input schema. The LLM replies with a *tool call*, for example `search_incidents` with `severity="Critical"`. LangChain runs the matching Python function and gives the result back to the LLM, which writes the final answer.

**Why authentication is required.** Every database connection needs credentials that identify the caller and limit what the caller may do. The agent works only through the connection configured in `.env`. Without the password, nobody can read or change the data.

**Role of Pydantic.** Pydantic schemas check each tool input before the function runs. They reject unknown severities, empty required text and invalid IDs. They also return a clear reason to the agent, so the agent can correct the request.

**Architecture of this lab**

```text
User Natural Language Request
             |
             v
       LangChain Agent
             |
             v
       Tool Selection
             |
             v
       Pydantic Validation
             |
             v
      Custom Python Tool
             |
             v
   Authenticated PostgreSQL
             |
             v
       Query / Update
             |
             v
       Agent Response
```

**Key point:** the LLM never connects to the database. It cannot see the password and it cannot write SQL. It can only ask for one of five authorized tools, and the Python tool does the database work.

## Section 2 - Cloud Database Setup (Supabase)

Do these steps once, in your browser. Supabase screens change from time to time, so follow the wording you see.

1. **Create a free Supabase account** at supabase.com.
2. **Create a new project.** Name it (for example `it-ops-lab`), choose a region near you, and set a **strong database password**. Store the password in a password manager. You can reset it later under Project Settings > Database.
3. **Find the PostgreSQL connection details.** Open the project and click **Connect** near the top of the dashboard. Note the **host**, **port**, **database name**, **user** and **password** for the PostgreSQL connection.
4. **Choose a reachable connection type.** The direct connection can fail on networks without IPv6. If that happens, use the **Session pooler** details from the same dialog. Copy the user name exactly as shown, because pooler user names can include the project reference.
5. **Configure `.env`.** In this lab folder, copy `.env.example` to `.env` and paste your own values. Never share the `.env` file.
6. **Verify the connection** in Section 3.

**Two different credentials. Do not confuse them.**

| | Supabase project URL and API keys | PostgreSQL database credentials |
|---|---|---|
| What they authenticate | Supabase web and app APIs (REST, Auth) | A direct SQL connection with a database user and password |
| Used in this lab? | **No** | **Yes**: host, port, database name, user, password |

**SSL.** Supabase encrypts database connections. The notebook uses `sslmode=require`, which means "encrypt the connection and do not fall back to plain text". Do not turn this off.

**Least privilege.** Setup needs an owner account to create the table. The agent's everyday account should have fewer rights. Section 10 shows how to create that restricted account.

In [ ]:
# Standard library
import os
import sys
from datetime import datetime
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Literal, Optional

# Reads the .env file in this folder into environment variables (values are never printed)
from dotenv import load_dotenv

# PostgreSQL driver (psycopg 3)
import psycopg

# Pydantic validates tool inputs before any database work happens
from pydantic import BaseModel, ConfigDict, Field, ValidationError, model_validator

# LangChain: messages, prompts, tools, the chat model and the agent
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import StructuredTool
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

# Load the settings from your .env file (copy .env.example to .env first)
load_dotenv()
print("Found a .env file in this folder:", Path(".env").exists())
print("Python version:", sys.version.split()[0])


def is_real_value(name):
    """True when the variable is set and is not still a placeholder from .env.example."""
    value = os.getenv(name, "").strip()
    return bool(value) and not value.startswith("your_")


# Show only True or False for each setting. Values and passwords are never printed.
for name in ["SUPABASE_DB_HOST", "SUPABASE_DB_PORT", "SUPABASE_DB_NAME",
             "SUPABASE_DB_USER", "SUPABASE_DB_PASSWORD", "OPENAI_API_KEY"]:
    print(f"{name:22} set: {is_real_value(name)}")

# Database settings. This dictionary holds the password, so never print it.
DB_SETTINGS = {
    "host": os.getenv("SUPABASE_DB_HOST", "").strip(),
    "port": os.getenv("SUPABASE_DB_PORT", "5432").strip(),
    "dbname": os.getenv("SUPABASE_DB_NAME", "postgres").strip(),
    "user": os.getenv("SUPABASE_DB_USER", "").strip(),
    "password": os.getenv("SUPABASE_DB_PASSWORD", ""),
}

# Lab settings
OPENAI_READY = is_real_value("OPENAI_API_KEY")
CHAT_MODEL = os.getenv("OPENAI_MODEL", "").strip() if is_real_value("OPENAI_MODEL") else "gpt-4o-mini"
MAX_ITERATIONS = 8   # safety limit on agent steps
MAX_ROWS = 50        # safety limit on rows returned by one read

print("Chat model:", CHAT_MODEL)
print("\nPackage versions in this interpreter:")
for package in ["langchain", "langchain-core", "langchain-openai", "langchain-classic",
                "psycopg", "pydantic", "python-dotenv"]:
    try:
        print(f"  {package:20} {version(package)}")
    except PackageNotFoundError:
        print(f"  {package:20} NOT INSTALLED")

## Section 3 - Connect Python to PostgreSQL

This section opens **one authenticated connection** and checks that it works. The safety rules used here:

- Only `True` or `False` is printed for each setting. Values and the password are never printed.
- The connection is built from keyword arguments, so no connection string is printed.
- `sslmode="require"` encrypts the connection. `connect_timeout=10` stops a connection that hangs.
- The `with` blocks commit on success, roll back on error and close the connection. This is the cleanup step.
- When something fails, the notebook prints a short, safe message instead of the raw driver error. Raw driver errors can contain the host and user name.

**Expected output:** `CONNECTION OK: PostgreSQL ...`. If you see `CONNECTION FAILED`, read the message and check the Troubleshooting section of the README.

In [ ]:
class DatabaseError(Exception):
    """An error message that is safe to show: it never contains a host, user name or password."""


def connect_to_database():
    """Open ONE authenticated, encrypted connection using DB_SETTINGS."""
    return psycopg.connect(**DB_SETTINGS, sslmode="require", connect_timeout=10)


def safe_error(err):
    """Turn a driver error into a DatabaseError with a message that reveals no secrets."""
    if isinstance(err, psycopg.OperationalError):
        return DatabaseError("The database connection failed. Check the SUPABASE_DB_* values in .env (Section 2).")
    if err.sqlstate == "42501":   # insufficient_privilege
        return DatabaseError("Permission denied by the database (error code 42501). This account is not allowed "
                             "to do this. Table setup needs the owner account; see Section 10 (least privilege).")
    return DatabaseError(f"The database rejected the request (error code {err.sqlstate or 'not provided'}). "
                         "No change was made.")


def run_sql(sql, params=None, fetch=False):
    """Run ONE parameterized SQL statement.

    The values go in `params`, never inside the SQL text. fetch=True returns rows as dictionaries.
    fetch=False returns how many rows changed.
    """
    try:
        with connect_to_database() as conn:          # commits on success, rolls back on error
            with conn.cursor() as cur:
                cur.execute(sql, params)
                if fetch:
                    columns = [col.name for col in cur.description]
                    return [dict(zip(columns, row)) for row in cur.fetchall()]
                return cur.rowcount
    except psycopg.Error as err:
        raise safe_error(err) from None


def test_connection():
    """Connect, read the server version and print a clear success or failure message."""
    try:
        row = run_sql("SELECT version() AS server_version", fetch=True)[0]
    except DatabaseError as err:
        print("CONNECTION FAILED:", err)
        return False
    print("CONNECTION OK:", row["server_version"].split(" on ")[0])
    return True


CONNECTED = test_connection()

## Section 4 - Create the IT Operations Table

**CREATE TABLE** defines the columns and their types:
- `incident_id SERIAL PRIMARY KEY`: a unique number the database generates for each row. The primary key identifies the row.
- `VARCHAR(n)` is text with a maximum length. `TEXT` has no fixed limit.
- `NOT NULL` means the column must always have a value.
- `CHECK (...)` limits a column to a list of allowed values. The database rejects anything else, even when a program forgets to check.
- `created_at ... DEFAULT NOW()` fills in the current time when no time is given.
- `IF NOT EXISTS` makes the statement safe to run again. It does nothing when the table already exists.

**Sample data.** The next cells define 15 fictional incidents for an IT operations team (servers, databases, networks and applications). Their IDs are 1 to 15, so later examples always refer to the same incidents. The insert runs **only when the table is empty**. Running it again prints a message and skips the insert.

In [ ]:
CREATE_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS incidents (
    incident_id     SERIAL PRIMARY KEY,
    title           VARCHAR(200) NOT NULL,
    description     TEXT NOT NULL,
    category        VARCHAR(50) NOT NULL
                    CHECK (category IN ('Network', 'Server', 'Database', 'Application')),
    severity        VARCHAR(20) NOT NULL
                    CHECK (severity IN ('Critical', 'High', 'Medium', 'Low')),
    status          VARCHAR(30) NOT NULL
                    CHECK (status IN ('Open', 'In Progress', 'Resolved')),
    assigned_team   VARCHAR(100) NOT NULL,
    affected_system VARCHAR(100) NOT NULL,
    created_at      TIMESTAMP NOT NULL DEFAULT NOW()
)
"""

run_sql(CREATE_TABLE_SQL)
print("Table 'incidents' is ready. It is created only when it does not already exist.")

In [ ]:
# 15 fictional incidents. Tuple order matches the INSERT in the next cell.
# (incident_id, title, description, category, severity, status, assigned_team, affected_system, created_at)
SAMPLE_INCIDENTS = [
    (1, "Application server CPU utilization is high",
     "CPU on APP-SRV-03 has stayed above 90% for 40 minutes. Overnight batch jobs are queued.",
     "Server", "High", "In Progress", "Infrastructure Team", "APP-SRV-03 (Order Processing)",
     datetime(2026, 11, 19, 8, 5)),
    (2, "Database connection timeout",
     "Order API connections to the orders database time out after 30 seconds during peak load.",
     "Database", "Critical", "Open", "Database Support Team", "Orders Database",
     datetime(2026, 11, 19, 9, 40)),
    (3, "Network connectivity failure between data centers",
     "About 40% packet loss on the WAN link between the primary and secondary data centers. Replication is delayed.",
     "Network", "Critical", "Open", "Network Operations", "DC1-DC2 WAN Link",
     datetime(2026, 11, 19, 10, 15)),
    (4, "Disk utilization exceeded threshold",
     "The data volume on FILE-SRV-02 is 92% used. The alert threshold is 85%.",
     "Server", "Medium", "Open", "Infrastructure Team", "FILE-SRV-02",
     datetime(2026, 11, 19, 10, 50)),
    (5, "Authentication service unavailable",
     "Users cannot sign in to internal applications. The login service returns HTTP 503 errors.",
     "Application", "Critical", "Resolved", "Identity Team", "Single Sign-On Service",
     datetime(2026, 11, 18, 16, 20)),
    (6, "Backup job failed for orders database",
     "The nightly backup stopped with a storage write error. The last successful backup was on 17 Nov.",
     "Database", "High", "Open", "Database Support Team", "Orders Database",
     datetime(2026, 11, 19, 6, 30)),
    (7, "Application response time degraded",
     "Customer portal page load time rose by about 2 seconds after the 18 Nov release.",
     "Application", "Medium", "In Progress", "Application Support", "Customer Portal",
     datetime(2026, 11, 18, 12, 0)),
    (8, "VPN gateway intermittent disconnects",
     "Remote users are disconnected from the VPN every 15 to 20 minutes.",
     "Network", "High", "Open", "Network Operations", "Remote Access VPN",
     datetime(2026, 11, 19, 11, 5)),
    (9, "TLS certificate expiring soon",
     "The certificate for the intranet portal expires in 7 days.",
     "Server", "Low", "Open", "Infrastructure Team", "Intranet Portal",
     datetime(2026, 11, 19, 7, 45)),
    (10, "Print queue stuck on Finance floor",
     "Print jobs on the Finance floor are stuck. Restarting the print spooler did not help.",
     "Server", "Low", "Resolved", "Service Desk", "Print Server PRN-01",
     datetime(2026, 11, 17, 14, 30)),
    (11, "Reporting queries exceed 10 minutes",
     "Monthly sales report queries run slowly. A missing index is suspected.",
     "Database", "Medium", "In Progress", "Database Support Team", "Sales Data Warehouse",
     datetime(2026, 11, 18, 15, 10)),
    (12, "Firewall rule change caused outage",
     "A rule change blocked HR application traffic to the payroll API for 25 minutes.",
     "Network", "High", "Resolved", "Network Operations", "Perimeter Firewall",
     datetime(2026, 11, 17, 22, 10)),
    (13, "Error rate spike after deployment",
     "HTTP 500 errors on the Order API rose to 6% after release 4.8.2.",
     "Application", "Critical", "In Progress", "Application Support", "Order API",
     datetime(2026, 11, 19, 9, 55)),
    (14, "Outbound email delivery delayed",
     "Emails sent from the CRM take up to 20 minutes to be delivered.",
     "Application", "Medium", "Open", "Messaging Team", "Mail Relay",
     datetime(2026, 11, 19, 12, 20)),
    (15, "HR database storage nearing capacity",
     "HR database storage is 88% full. Growth is about 2% per week.",
     "Database", "Medium", "Open", "Database Support Team", "HR Database",
     datetime(2026, 11, 19, 13, 0)),
]
print(f"{len(SAMPLE_INCIDENTS)} sample incidents defined (not inserted yet).")

In [ ]:
INSERT_INCIDENT_SQL = """
INSERT INTO incidents (incident_id, title, description, category, severity, status,
                       assigned_team, affected_system, created_at)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""


def seed_sample_data():
    """Insert the sample incidents, but only when the table is empty. All rows go in ONE transaction."""
    existing = run_sql("SELECT COUNT(*) AS total FROM incidents", fetch=True)[0]["total"]
    if existing > 0:
        print(f"Skipped: the table already has {existing} incidents. No sample data was inserted.")
        return
    try:
        with connect_to_database() as conn:
            with conn.cursor() as cur:
                cur.executemany(INSERT_INCIDENT_SQL, SAMPLE_INCIDENTS)
                # Move the SERIAL counter past the explicit IDs, so the next new incident gets ID 16
                cur.execute("SELECT setval(pg_get_serial_sequence('incidents', 'incident_id'), "
                            "(SELECT MAX(incident_id) FROM incidents))")
    except psycopg.Error as err:
        raise safe_error(err) from None
    print(f"Inserted {len(SAMPLE_INCIDENTS)} sample incidents.")


seed_sample_data()

## Section 4 (continued) - Basic SQL Queries

The agent relies on four kinds of query:

- **View all rows:** `SELECT ... FROM incidents ORDER BY incident_id`
- **Filter by severity or status:** `WHERE severity = %s` or `WHERE status = %s`. The `%s` is a **placeholder**. The value is sent separately from the SQL text. Section 5 explains why this matters.
- **Count:** `SELECT COUNT(*) ...`

**Expected output:** 15 incidents in total, 4 Critical, and 8 Open.

In [ ]:
def print_table(rows):
    """Print incident rows as a simple text table."""
    if not rows:
        print("(no rows)")
        return
    print(f"{'ID':>3}  {'Severity':<9} {'Status':<12} {'Category':<12} {'Team':<22} Title")
    for r in rows:
        print(f"{r['incident_id']:>3}  {r['severity']:<9} {r['status']:<12} "
              f"{r['category']:<12} {r['assigned_team']:<22} {r['title']}")


# 1. View all incidents
print("1) All incidents:")
print_table(run_sql("SELECT incident_id, title, category, severity, status, assigned_team "
                    "FROM incidents ORDER BY incident_id", fetch=True))

# 2. Filter by severity. The value ("Critical",) is passed separately from the SQL.
print("\n2) Critical incidents:")
print_table(run_sql("SELECT incident_id, title, category, severity, status, assigned_team "
                    "FROM incidents WHERE severity = %s ORDER BY incident_id", ("Critical",), fetch=True))

# 3. Filter by status
print("\n3) Open incidents:")
print_table(run_sql("SELECT incident_id, title, category, severity, status, assigned_team "
                    "FROM incidents WHERE status = %s ORDER BY incident_id", ("Open",), fetch=True))

# 4. Count
total = run_sql("SELECT COUNT(*) AS total FROM incidents", fetch=True)[0]["total"]
print(f"\n4) Total incidents: {total}")

## Section 5 - Simple Database Functions

Each function does **one** database job. The agent never calls these functions directly. It calls the LangChain tools in Section 7, and those tools call these functions only after validation.

**Parameterized SQL prevents SQL injection.** Compare two ways of building a query:

```python
# UNSAFE - never do this. User text becomes part of the SQL command itself.
sql = f"SELECT * FROM incidents WHERE severity = '{severity}'"

# SAFE - the SQL text is fixed. The value travels separately as a parameter.
run_sql("SELECT * FROM incidents WHERE severity = %s", (severity,), fetch=True)
```

If someone types `Critical' OR '1'='1` into the unsafe version, the query returns every incident. In the safe version, the database treats the whole input as one literal value, so it matches no rows. The code block above is for explanation only and is not run.

**Functions in this section**
- Read functions (safe to run any time): `get_all_incidents`, `search_incidents_by_severity`, `search_incidents_by_status`, `get_incident_by_id`, `get_incident_summary`.
- Write functions (defined here, **not run** yet): `add_incident`, `update_incident_status`. The agent runs them only after the confirmation steps in Sections 7 and 9.

In [ ]:
SELECT_COLUMNS_SQL = "SELECT incident_id, title, category, severity, status, assigned_team FROM incidents"


def get_all_incidents():
    """Return up to MAX_ROWS incidents, oldest first."""
    return run_sql(SELECT_COLUMNS_SQL + " ORDER BY incident_id LIMIT %s", (MAX_ROWS,), fetch=True)


def search_incidents_by_severity(severity):
    """Return incidents with one severity: Critical, High, Medium or Low."""
    return run_sql(SELECT_COLUMNS_SQL + " WHERE severity = %s ORDER BY incident_id", (severity,), fetch=True)


def search_incidents_by_status(status):
    """Return incidents with one status: Open, In Progress or Resolved."""
    return run_sql(SELECT_COLUMNS_SQL + " WHERE status = %s ORDER BY incident_id", (status,), fetch=True)


def get_incident_by_id(incident_id):
    """Return one incident as a dictionary, or None when that ID does not exist."""
    rows = run_sql(
        "SELECT incident_id, title, category, severity, status, assigned_team, affected_system "
        "FROM incidents WHERE incident_id = %s",
        (incident_id,), fetch=True)
    return rows[0] if rows else None


def get_incident_summary():
    """Count incidents by severity and by status."""
    by_severity = run_sql("SELECT severity, COUNT(*) AS total FROM incidents GROUP BY severity", fetch=True)
    by_status = run_sql("SELECT status, COUNT(*) AS total FROM incidents GROUP BY status", fetch=True)
    return {
        "by_severity": {row["severity"]: row["total"] for row in by_severity},
        "by_status": {row["status"]: row["total"] for row in by_status},
    }


# Quick checks of the read functions (they change nothing)
print("get_all_incidents():", len(get_all_incidents()), "rows")
print("search_incidents_by_severity('Critical'):", len(search_incidents_by_severity("Critical")), "rows")
print("search_incidents_by_status('Open'):", len(search_incidents_by_status("Open")), "rows")
print("get_incident_summary():", get_incident_summary())

In [ ]:
def add_incident(title, description, category, severity, assigned_team, affected_system):
    """Insert one new incident with status 'Open'. The database generates the incident_id and returns it."""
    rows = run_sql(
        """
        INSERT INTO incidents (title, description, category, severity, status,
                               assigned_team, affected_system, created_at)
        VALUES (%s, %s, %s, %s, 'Open', %s, %s, NOW())
        RETURNING incident_id
        """,
        (title, description, category, severity, assigned_team, affected_system),
        fetch=True,
    )
    return rows[0]["incident_id"]


def update_incident_status(incident_id, status):
    """Change the status of ONE incident. Returns the number of rows changed (1 means success)."""
    return run_sql(
        "UPDATE incidents SET status = %s WHERE incident_id = %s",
        (status, incident_id),
    )


print("add_incident() and update_incident_status() are defined. The agent runs them only after confirmation.")

## Section 6 - Pydantic Schemas for Tool Validation

**Pydantic** turns a set of rules into a class. Python checks every value against the rules before a function uses it.

| Schema | Used by | Rules |
|---|---|---|
| `IncidentSearchInput` | `search_incidents` | Exactly one of `severity` or `status`, and only allowed values |
| `IncidentCreateInput` | `create_incident` | Required text must not be empty (surrounding spaces are removed first). `category` and `severity` must be in the allowed lists. Length limits match the table columns. |
| `IncidentUpdateInput` | `update_incident_status` | `incident_id` must be a positive whole number. `status` must be in the allowed list. |

Both write schemas also have `confirmed` (default `False`). It is the write gate explained in Section 7. `strict=True` means it must be a real `true` or `false`, not the text `"yes"`.

Pydantic validates the **input**. It does not replace database permissions or the database's own CHECK constraints (see Section 10).

In [ ]:
# Allowed values. They match the CHECK constraints in the table.
Severity = Literal["Critical", "High", "Medium", "Low"]
Status = Literal["Open", "In Progress", "Resolved"]
Category = Literal["Network", "Server", "Database", "Application"]


class IncidentSearchInput(BaseModel):
    """Input for search_incidents: give exactly one of severity or status."""
    severity: Optional[Severity] = None
    status: Optional[Status] = None

    @model_validator(mode="after")
    def exactly_one_filter(self):
        if (self.severity is None) == (self.status is None):
            raise ValueError("Give exactly one of: severity or status.")
        return self


class IncidentCreateInput(BaseModel):
    """Input for create_incident. Text fields are trimmed and must not be empty."""
    model_config = ConfigDict(str_strip_whitespace=True)

    title: str = Field(min_length=1, max_length=200)
    description: str = Field(min_length=1)
    category: Category
    severity: Severity
    assigned_team: str = Field(min_length=1, max_length=100)
    affected_system: str = Field(min_length=1, max_length=100)
    # The write gate. False = validate and preview only. True = save (see Section 7).
    confirmed: bool = Field(default=False, strict=True)


class IncidentUpdateInput(BaseModel):
    """Input for update_incident_status. The ID must be a positive whole number."""
    incident_id: int = Field(gt=0)
    status: Status
    confirmed: bool = Field(default=False, strict=True)


print("Pydantic schemas defined: IncidentSearchInput, IncidentCreateInput, IncidentUpdateInput")

In [ ]:
def try_build(label, model, **fields):
    """Build a Pydantic input and print ACCEPTED, or REJECTED with the reasons."""
    print(f"\n{label}")
    try:
        item = model(**fields)
        print("   ACCEPTED ->", item.model_dump())
    except ValidationError as err:
        print("   REJECTED ->")
        for problem in err.errors():
            field = problem["loc"][0] if problem["loc"] else "request"
            print(f"      {field}: {problem['msg']}")


good_fields = dict(
    title="Payment database connection timeouts",
    description="Payment database is experiencing connection timeouts.",
    category="Database",
    severity="High",
    assigned_team="Database Support Team",
    affected_system="Payments Database",
)

try_build("1) VALID create request", IncidentCreateInput, **good_fields)
try_build("2) INVALID severity ('Urgent' is not allowed)", IncidentCreateInput,
          **{**good_fields, "severity": "Urgent"})
try_build("3) INVALID status for an update ('Done' is not allowed)", IncidentUpdateInput,
          incident_id=3, status="Done")
try_build("4) MISSING required field (affected_system left out)", IncidentCreateInput,
          **{k: v for k, v in good_fields.items() if k != "affected_system"})
try_build("5) INVALID incident ID (must be a positive whole number)", IncidentUpdateInput,
          incident_id=0, status="Resolved")

## Section 7 - Build Custom LangChain Tools

A **tool** is a Python function that LangChain describes to the LLM. Each tool has three parts:
1. **Name**: what the LLM calls the tool.
2. **Description**: when to use it. The LLM reads this text to choose a tool.
3. **Input schema** (`args_schema`): the Pydantic model that validates the arguments.

We build the tools with `StructuredTool.from_function`. It is the same tool class that the `@tool` decorator creates, and it lets us return Pydantic rejection reasons to the agent instead of a generic error.

| Tool name | Purpose | Writes data? |
|---|---|---|
| `get_all_incidents` | Retrieve incidents (up to 50, oldest first) | No |
| `search_incidents` | Search by one severity or one status | No |
| `get_incident_summary` | Counts by severity and status, plus unresolved Critical incidents | No |
| `create_incident` | Insert a new incident with status Open | **Yes**: confirmation required |
| `update_incident_status` | Change the status of one incident | **Yes**: confirmation required |

**Write confirmation: two locks.** Each write uses two steps:
1. The agent calls the tool with `confirmed=false`. The tool validates the input and returns a preview. Nothing is saved.
2. The agent shows the preview and asks the learner. Only after the learner says yes does the agent call the tool again with `confirmed=true`.

The code also checks two things. The confirmed call must match the previewed values exactly. The **learner's own latest message** must be a clear yes. The model cannot set that second condition, so the agent cannot write without the learner's approval.

This is a teaching guardrail. A production system would add a real approval step outside the chat.

**No SQL tool.** No tool accepts SQL text, and no tool deletes or drops data. The agent can only do what these five tools do.

In [ ]:
import functools


def format_rows(rows):
    """Turn incident rows into short text lines that the agent can read."""
    if not rows:
        return "No matching incidents were found."
    lines = [f"Found {len(rows)} incident(s):"]
    for r in rows:
        lines.append(f"- #{r['incident_id']} [{r['severity']}] {r['title']} | status: {r['status']} | "
                     f"category: {r['category']} | team: {r['assigned_team']}")
    return "\n".join(lines)


def validation_message(err):
    """Explain rejected tool inputs in words the agent can act on. Nothing runs when inputs are rejected."""
    problems = []
    for problem in err.errors():
        field = problem["loc"][0] if problem["loc"] else "request"
        message = problem["msg"].replace("Value error, ", "")
        problems.append(f"{field}: {message}")
    return "INPUT REJECTED - nothing was run. " + " | ".join(problems)


def make_tool(func, name, description, args_schema=None):
    """Wrap a Python function as a LangChain tool. Errors go back to the agent as text, so the loop keeps running."""
    @functools.wraps(func)
    def guarded(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except DatabaseError as err:          # our own safe message (no secrets)
            return f"DATABASE ERROR - {err}"
        except Exception as err:              # anything unexpected: show only the error type
            return (f"TOOL ERROR ({type(err).__name__}) - the operation did not complete. "
                    "Nothing was confirmed as saved.")

    return StructuredTool.from_function(
        func=guarded,
        name=name,
        description=description,
        args_schema=args_schema,
        handle_validation_error=validation_message,   # Pydantic reasons go back to the agent
    )


# ---- Read tools -------------------------------------------------------------
def tool_get_all_incidents():
    """Python function behind the get_all_incidents tool."""
    return format_rows(get_all_incidents())


def tool_search_incidents(severity: Optional[str] = None, status: Optional[str] = None):
    """Python function behind the search_incidents tool. The schema guarantees exactly one filter."""
    if severity is not None:
        return format_rows(search_incidents_by_severity(severity))
    return format_rows(search_incidents_by_status(status))


def tool_get_incident_summary():
    """Python function behind the get_incident_summary tool."""
    summary = get_incident_summary()
    by_sev, by_stat = summary["by_severity"], summary["by_status"]
    critical_unresolved = [r for r in search_incidents_by_severity("Critical") if r["status"] != "Resolved"]
    lines = [
        f"Total incidents: {sum(by_sev.values())}",
        "By severity: " + ", ".join(f"{s} {by_sev.get(s, 0)}" for s in ["Critical", "High", "Medium", "Low"]),
        "By status: " + ", ".join(f"{s} {by_stat.get(s, 0)}" for s in ["Open", "In Progress", "Resolved"]),
    ]
    if critical_unresolved:
        lines.append("Unresolved Critical incidents: " +
                     "; ".join(f"#{r['incident_id']} {r['title']} ({r['status']})" for r in critical_unresolved))
    else:
        lines.append("No Critical incidents are unresolved.")
    return "\n".join(lines)


# ---- Write tools: the confirmation state is set by the app, not by the model ----
PENDING_PREVIEW = None     # the exact write the learner was last shown as a preview
LEARNER_SAID_YES = False   # True only when the learner's own latest message is a clear yes


def check_write_allowed(preview_key):
    """Return None when the write may run. Otherwise return a BLOCKED message."""
    if PENDING_PREVIEW != preview_key:
        return ("BLOCKED - nothing was saved. These values do not match a preview the user has seen. "
                "Show the preview first (confirmed=false).")
    if not LEARNER_SAID_YES:
        return "BLOCKED - nothing was saved. The user has not said yes to this preview in their latest message."
    return None


def tool_create_incident(title: str, description: str, category: str, severity: str,
                         assigned_team: str, affected_system: str, confirmed: bool = False):
    """Python function behind the create_incident tool."""
    global PENDING_PREVIEW
    preview_key = ("create", title, description, category, severity, assigned_team, affected_system)
    if not confirmed:
        PENDING_PREVIEW = preview_key
        return ("PREVIEW ONLY - NOTHING WAS SAVED. Validation passed. Proposed incident: "
                f"title '{title}'; description '{description}'; category {category}; severity {severity}; "
                f"team {assigned_team}; affected system {affected_system}; status Open. "
                "Ask the user for an explicit yes or no.")
    blocked = check_write_allowed(preview_key)
    if blocked:
        return blocked
    PENDING_PREVIEW = None
    new_id = add_incident(title, description, category, severity, assigned_team, affected_system)
    saved = get_incident_by_id(new_id)   # read the row back to verify the insert
    if saved is None:
        return "NOT CONFIRMED - the insert returned an ID, but the record could not be read back."
    return (f"SAVED. The database confirmed new incident #{saved['incident_id']} "
            f"(severity {saved['severity']}, status {saved['status']}).")


def tool_update_incident_status(incident_id: int, status: str, confirmed: bool = False):
    """Python function behind the update_incident_status tool."""
    global PENDING_PREVIEW
    preview_key = ("update", incident_id, status)
    current = get_incident_by_id(incident_id)
    if current is None:
        return f"NOT FOUND - no incident with ID {incident_id} exists. Nothing was changed."
    if not confirmed:
        PENDING_PREVIEW = preview_key
        return (f"PREVIEW ONLY - NOTHING WAS CHANGED. Incident #{incident_id} ('{current['title']}') is currently "
                f"'{current['status']}'. Proposed new status: '{status}'. Ask the user for an explicit yes or no.")
    blocked = check_write_allowed(preview_key)
    if blocked:
        return blocked
    PENDING_PREVIEW = None
    rows_changed = update_incident_status(incident_id, status)
    after = get_incident_by_id(incident_id)   # verify against the database
    if rows_changed == 1 and after is not None and after["status"] == status:
        return f"UPDATED. The database confirmed incident #{incident_id} is now '{status}' (1 row changed)."
    return f"NOT CONFIRMED - the database does not show the new status for incident #{incident_id}."


# ---- Register the five tools ------------------------------------------------
ALL_TOOLS = [
    make_tool(tool_get_all_incidents, "get_all_incidents",
              "Retrieve up to 50 incidents, oldest first. Use when the user asks to see all incidents."),
    make_tool(tool_search_incidents, "search_incidents",
              "Find incidents by ONE severity (Critical, High, Medium, Low) or ONE status (Open, In Progress, Resolved).",
              IncidentSearchInput),
    make_tool(tool_get_incident_summary, "get_incident_summary",
              "Count incidents by severity and by status, and list unresolved Critical incidents. "
              "Use for overview or summary questions."),
    make_tool(tool_create_incident, "create_incident",
              "Create a new incident with status Open. First call with confirmed=false to validate and preview. "
              "Call with confirmed=true ONLY after the user has said yes to that preview.",
              IncidentCreateInput),
    make_tool(tool_update_incident_status, "update_incident_status",
              "Change the status of ONE existing incident. First call with confirmed=false to check it and preview. "
              "Call with confirmed=true ONLY after the user has said yes.",
              IncidentUpdateInput),
]

print(f"{len(ALL_TOOLS)} custom tools registered:\n")
for t in ALL_TOOLS:
    print(f"- {t.name}")
    print(f"    inputs : {list(t.args.keys()) or 'none'}")
    print(f"    purpose: {t.description}")

In [ ]:
# Safety check: no tool name suggests raw SQL, dropping, truncating or deleting
risky_words = ("sql", "query", "drop", "delete", "truncate")
assert not any(word in t.name for t in ALL_TOOLS for word in risky_words), "A tool could run raw SQL"
print("Check passed: no tool accepts raw SQL or can drop, truncate or delete data.")

## Section 8 - Build the LangChain SQL Database Agent

**Components**
- **ChatOpenAI**: the LLM. `temperature=0` keeps the answers consistent.
- **Prompt**: the system rules, the chat history (earlier turns, so the agent remembers a pending confirmation), the learner's request, and the agent scratchpad (tool steps).
- **`create_tool_calling_agent`**: attaches the five tools to the LLM and the prompt.
- **`AgentExecutor`**: runs the loop. It calls the LLM, runs the chosen tool, returns the result to the LLM and repeats until there is a final answer or `MAX_ITERATIONS` is reached. `verbose=True` prints each step.

These APIs come from `langchain-classic` 1.0.8, the same as Module 8 Lab 1. This keeps the LangChain versions consistent.

**System prompt.** It tells the agent to use only its tools, never invent records, ask for missing required fields, ask for explicit confirmation before a write, and report success only when the tool says the database confirmed it. The prompt contains no passwords and no connection details.

In [ ]:
SYSTEM_PROMPT = """You are an IT Operations assistant for an incident database.

Rules you must follow:
1. Understand the user's request, then choose the right tool from the tools you were given.
2. Use ONLY those tools. You cannot run SQL, and you cannot see or change database settings or passwords.
3. Give each tool only the inputs the user provided or clearly stated. Use the exact allowed values:
   severity = Critical, High, Medium or Low. status = Open, In Progress or Resolved.
   category = Network, Server, Database or Application.
4. Do NOT invent details. If a required field is missing (for example affected_system or assigned_team), ask the user for it.
5. Never invent incident records, IDs or query results. Report only what the tools return.
6. Writes need confirmation. For create_incident and update_incident_status:
   a. First call the tool with confirmed=false. This validates the input and shows a preview. Nothing is saved.
   b. Show the user a short summary of the preview and ask: "Shall I save this? (yes/no)".
   c. Call the tool again with confirmed=true ONLY after the user explicitly says yes in their latest message. Use the same values as the preview.
7. Say that an insert or update succeeded ONLY if the tool result says SAVED or UPDATED.
8. If a tool returns an error, explain it in plain language and suggest the next step. Never reveal technical connection details.
9. Present results in business-friendly language: short sentences, with Critical and Open items first.
"""

agent_executor = None
if OPENAI_READY:
    llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        MessagesPlaceholder("chat_history"),       # earlier turns, for example a pending confirmation
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),   # the tool steps for this request
    ])
    agent = create_tool_calling_agent(llm, ALL_TOOLS, prompt)
    agent_executor = AgentExecutor(
        agent=agent,
        tools=ALL_TOOLS,
        verbose=True,                    # print every step (the tool trace)
        max_iterations=MAX_ITERATIONS,   # safety limit on the loop
        return_intermediate_steps=True,  # keep the list of tools that were called
        handle_parsing_errors=True,
    )
    print("Agent ready. Model:", CHAT_MODEL, "| max steps:", MAX_ITERATIONS)
else:
    print("Agent NOT created: OPENAI_API_KEY is missing in .env. Sections 1 to 7 still run.")

YES_WORDS = ("yes", "y", "yep", "yeah", "confirm", "approve", "go ahead", "ok", "okay", "sure")


def user_said_yes(text):
    """True only when the learner's own message starts with a clear yes. The app checks this, not the model."""
    cleaned = text.strip().lower()
    return any(cleaned == w or cleaned.startswith((w + " ", w + ",", w + ".", w + "!")) for w in YES_WORDS)


def chat_with_agent(user_text, history):
    """Send one message to the agent. `history` keeps earlier turns so a pending confirmation is remembered."""
    global LEARNER_SAID_YES
    if agent_executor is None:
        raise RuntimeError("The agent is not ready. Add OPENAI_API_KEY to .env, restart the kernel and run Section 3.")
    LEARNER_SAID_YES = user_said_yes(user_text)   # set by the app from the learner's words
    result = agent_executor.invoke({"input": user_text, "chat_history": history})
    reply = result["output"]
    tools_used = [step[0].tool for step in result["intermediate_steps"]]
    history.extend([HumanMessage(content=user_text), AIMessage(content=reply)])
    print("\nTOOLS CALLED:", tools_used or "none")
    print("AGENT:", reply)
    return reply

## Section 9 - Natural Language Database Operations

Each scenario runs against your **real** database, through the real tools. Run each scenario cell **once**. Running a write scenario again creates another incident or repeats an update.

| Scenario | What you type | Expected behavior |
|---|---|---|
| 1 | "Show me all open IT incidents." | Agent calls `search_incidents` with status Open. Expected: **8** incidents (#2, #3, #4, #6, #8, #9, #14, #15). |
| 2 | "Show me all critical incidents." | Agent calls `search_incidents` with severity Critical. Expected: **4** incidents (#2, #3, #5, #13). |
| 3 | A high-severity database incident for payment timeouts, assigned to the Database Support Team | Agent asks for the missing **affected system** and does not guess it. It then shows a preview and asks yes or no. It saves only after your yes and reports the new ID from the database (expected **#16**). |
| 4 | "Update incident 3 to In Progress." | Agent checks incident 3 (currently Open), shows a preview and asks for confirmation. After your yes it updates the incident and reports what the database now shows. |
| 5 | "Give me a summary of the current IT incident situation." | Agent calls `get_incident_summary` and highlights the unresolved Critical incidents. |

Scenarios 3 and 4 change the data. After Scenario 3 the total is one higher (16), and after Scenario 4 incident 3 is In Progress.

In [ ]:
# Scenario 1: a read request. Expected tool: search_incidents(status="Open").
chat_with_agent("Show me all open IT incidents.", [])

# Scenario 2: a read request. Expected tool: search_incidents(severity="Critical").
chat_with_agent("Show me all critical incidents.", [])

In [ ]:
# Scenario 3: a write request with a missing field. Three learner turns share one history.
history_s3 = []
chat_with_agent("Create a high-severity database incident because the payment database is experiencing "
                "connection timeouts. Assign it to the Database Support Team.", history_s3)
chat_with_agent("The affected system is the Payments Database.", history_s3)   # answers the missing field
chat_with_agent("Yes, please save it.", history_s3)                             # explicit confirmation

In [ ]:
# Scenario 4: an update request. The first turn previews the change; the second turn confirms it.
history_s4 = []
chat_with_agent("Update incident 3 to In Progress.", history_s4)
chat_with_agent("Yes, go ahead.", history_s4)

In [ ]:
# Scenario 5: a business summary. Expected tool: get_incident_summary.
chat_with_agent("Give me a summary of the current IT incident situation.", [])

## Section 10 - Authentication and Security

**The practical security model**

| Control | How this lab implements it |
|---|---|
| 1. The agent never sees passwords | Passwords live only in `.env` and in `DB_SETTINGS`. The system prompt and tool outputs never contain them. |
| 2. No arbitrary SQL | Five named tools. No tool accepts SQL text. |
| 3. No dropping or truncating tables | No tool runs DDL. The restricted role below has no right to drop or truncate. |
| 4. No deleting records | No delete tool. The restricted role has no DELETE right. |
| 5. Writes need explicit confirmation | Preview first, then the learner's yes (two locks in Section 7). |
| 6. Updates affect only the requested incident | `WHERE incident_id = %s`. The tool checks that the incident exists and that exactly one row changed. |
| 7. Invalid inputs are rejected | Pydantic rejects them before any SQL runs. The reason goes back to the agent. |
| 8. Database errors don't expose secrets | `safe_error()` replaces driver text with generic messages, and `make_tool()` returns them to the agent as text. |

**Pydantic validates inputs. It does not replace database permissions.** A bug in validation, or a permission that is too broad, would still matter. Use layers: validation (Pydantic), database constraints (`CHECK`) and database permissions (least privilege).

**Why unrestricted SQL execution by an LLM is risky.** A model can misread a request, can be steered by text inside the data (prompt injection), or can be given a bad example. With a general SQL tool it could read sensitive data, delete rows or drop a table. The lab's tools limit what is possible in advance.

**Least-privilege account (recommended).** Run the SQL below in the Supabase **SQL Editor** while signed in as the project owner. Replace the placeholder password first. Keep the password private.

```sql
CREATE ROLE it_agent_app LOGIN PASSWORD 'REPLACE_WITH_A_STRONG_PASSWORD';
GRANT CONNECT ON DATABASE postgres TO it_agent_app;
GRANT USAGE ON SCHEMA public TO it_agent_app;
GRANT SELECT, INSERT, UPDATE ON TABLE incidents TO it_agent_app;
GRANT USAGE, SELECT ON SEQUENCE incidents_incident_id_seq TO it_agent_app;
```

**To practice it:** after setup, set `SUPABASE_DB_USER` and `SUPABASE_DB_PASSWORD` in `.env` to the `it_agent_app` account (with the pooler, the user name may need the project reference, as the README explains). Restart the kernel, then run Section 3 and Scenario 1 again. Reads, new incidents and status updates still work. Any attempt to delete, drop or truncate is refused by the database itself. Switch back to the owner account if you need to re-run the table setup cells.

In [ ]:
# 1. The tool list: no tool accepts SQL text or can delete or drop data
print("Tools available to the agent:", [t.name for t in ALL_TOOLS])

# 2. The agent's instructions must not contain the database password (only True or False is printed)
if DB_SETTINGS["password"]:
    print("Database password appears in the agent prompt:", DB_SETTINGS["password"] in SYSTEM_PROMPT)

# 3. The database's own CHECK constraints: a second safety layer below Pydantic (read-only query)
checks = run_sql(
    "SELECT conname FROM pg_constraint WHERE conrelid = 'incidents'::regclass AND contype = 'c' ORDER BY conname",
    fetch=True)
print("CHECK constraints on the incidents table:", [row["conname"] for row in checks])

# 4. Current totals (read-only)
print("Incidents in the database now:", run_sql("SELECT COUNT(*) AS total FROM incidents", fetch=True)[0]["total"])

## Section 11 - Hands-on Learner Exercises

Try each exercise yourself before you open the expected outcome. Replace the `TODO` text in the next cell with your own request, then run the cell. Exercises 2 and 3 need a second turn to confirm.

- **Exercise 1 - Incident search.** Ask the agent to find all High-severity incidents.
- **Exercise 2 - Incident creation.** Ask the agent to create a new Network incident and confirm the insertion. You choose the title, description, team and affected system. Answer the agent's questions, then reply yes to the preview.
- **Exercise 3 - Incident update.** Ask the agent to update an existing incident to Resolved and verify the result. Choose an incident that is not yet Resolved.

<details>
<summary><b>Expected outcome: Exercise 1</b></summary>

The agent calls `search_incidents` with severity High. Expected: **4** incidents (#1, #6, #8, #12). The answer should list them, with the unresolved ones first.

</details>

<details>
<summary><b>Expected outcome: Exercise 2</b></summary>

The agent asks for any missing field, shows a preview with status Open, and asks yes or no. After your yes it calls `create_incident` with `confirmed=true`. It reports the new incident ID only after the database confirms the insert. Check the new row with a count or a search.

</details>

<details>
<summary><b>Expected outcome: Exercise 3</b></summary>

The agent previews the change and shows the current status. After your yes it calls `update_incident_status` with `confirmed=true`. It then reports that the database now shows Resolved for that incident.

</details>

In [ ]:
# Replace each TODO text with your own request. Turn 2 is your reply to the agent (for example "yes").
exercise_1_turns = ["TODO: ask the agent to find all High-severity incidents"]
exercise_2_turns = ["TODO: ask the agent to create a new Network incident (give the details the agent asks for)",
                    "TODO: reply to the agent, or confirm with yes when it previews the incident"]
exercise_3_turns = ["TODO: ask the agent to update an existing incident to Resolved",
                    "TODO: confirm with yes when the agent previews the change"]

for exercise_name, turns in [("Exercise 1", exercise_1_turns),
                             ("Exercise 2", exercise_2_turns),
                             ("Exercise 3", exercise_3_turns)]:
    real_turns = [turn for turn in turns if not turn.startswith("TODO")]
    if not real_turns or turns[0].startswith("TODO"):
        print(f"{exercise_name}: skipped. Replace the TODO text first.")
        continue
    print(f"\n===== {exercise_name} =====")
    exercise_history = []
    for turn in real_turns:
        chat_with_agent(turn, exercise_history)

## Section 12 - Knowledge Check

Answer the five questions first. Then open the answer key.

**1. What is the main purpose of a SQL database agent?**
- A) Replace the database server
- B) Let an LLM choose authorized tools that run validated database operations and report the results
- C) Let the LLM write and run any SQL it likes
- D) Store database passwords inside the prompt

**2. What is the role of Pydantic schemas in this lab?**
- A) Encrypt the database connection
- B) Validate tool inputs (allowed values, required fields, lengths) before any database work happens
- C) Create the database tables
- D) Replace database permissions

**3. Why are authenticated database connections required?**
- A) To make the database public
- B) So that only identified, authorized credentials can connect, and the agent works only through them
- C) So that developers never have to write SQL
- D) To make queries run faster

**4. Why are parameterized SQL queries used?**
- A) They are shorter to write
- B) The values are sent separately from the SQL text, so user input cannot change the structure of the query
- C) They create indexes automatically
- D) They allow several tables in one query

**5. Why should an LLM not be given an arbitrary SQL execution tool?**
- A) It makes answers slower
- B) It could run destructive or unauthorized SQL, such as DROP, DELETE or reading sensitive data, which validation cannot fully control
- C) Databases cannot run SELECT statements
- D) LangChain does not allow it

<details>
<summary><b>Answer key</b></summary>

1. **B**
2. **B**
3. **B**
4. **B**
5. **B**

</details>

## Section 13 - Lab Summary

1. Create a cloud-hosted SQL database (Supabase).
2. Configure authenticated access (`.env`, SSL, a least-privilege account).
3. Connect Python to PostgreSQL (`psycopg`, safe error messages).
4. Create an IT Operations table (`CREATE TABLE`, constraints, sample data).
5. Implement database functions (parameterized SQL).
6. Define Pydantic validation schemas.
7. Convert the functions into custom LangChain tools, with a write gate.
8. Connect the tools to a LangChain agent (`create_tool_calling_agent` and `AgentExecutor`).
9. Perform natural language read and write operations, with confirmation.
10. Return business-friendly answers based on what the database confirmed.

**Enterprise applications.** The same pattern works for:
- **Incident management**: triage, status updates and escalation summaries.
- **Service requests**: create and track requests, with approval before a write.
- **Asset management**: look up assets and owners, and update records after approval.
- **Operations reporting**: summarize backlogs and trends using a read-only account.

The rule stays the same in every case: **the model proposes, validated tools execute, and database permissions decide what is allowed.**